In [3]:
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
df  = pd.read_csv("../data/wearables_daily.csv")
data = df[df["steps"]< 60000].dropna(subset = ["sleep_hours"])
data

,user_id,date,city,device,age,gender,steps,active_minutes,sleep_hours,deep_sleep_pct,resting_hr,hrv_ms,stress_score,calories,age_group,lat,lon
0,U001,2025-10-01,Berlin,Apple Watch,58,female,11208,91.0,6.24,12.8,68.0,38.7,56.0,2412.0,50-59,52.52,13.4
1,U001,2025-10-02,Berlin,Apple Watch,58,female,14360,123.0,5.86,17.7,65.0,38.6,56.0,2666.0,50-59,52.52,13.4
2,U001,2025-10-03,Berlin,Apple Watch,58,female,10323,81.0,6.95,15.6,65.0,31.5,56.0,2337.0,50-59,52.52,13.4
3,U001,2025-10-04,Berlin,Apple Watch,58,female,10048,87.0,7.06,24.8,65.0,30.5,41.0,2350.0,50-59,52.52,13.4
4,U001,2025-10-05,Berlin,Apple Watch,58,female,10675,98.0,6.76,12.8,64.0,39.0,59.0,2419.0,50-59,52.52,13.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91245,U250,2026-09-26,Amsterdam,Withings,59,male,10582,85.0,7.07,15.1,67.0,31.5,57.0,2363.0,50-59,52.37,4.9
91246,U250,2026-09-27,Amsterdam,Withings,59,male,7547,72.0,8.01,10.1,65.0,NaN,39.0,2190.0,50-59,52.37,4.9
91247,U250,2026-09-28,Amsterdam,Withings,59,male,21252,184.0,7.22,15.9,70.0,34.1,49.0,3186.0,50-59,52.37,4.9
91248,U250,2026-09-29,Amsterdam,Withings,59,male,20826,173.0,7.52,27.3,63.0,NaN,35.0,3125.0,50-59,52.37,4.9


In [6]:
data.info()

<class 'pandas.DataFrame'>
Index: 88456 entries, 0 to 91249
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         88456 non-null  str    
 1   date            88456 non-null  str    
 2   city            88456 non-null  str    
 3   device          88456 non-null  str    
 4   age             88456 non-null  int64  
 5   gender          88456 non-null  str    
 6   steps           88456 non-null  int64  
 7   active_minutes  88456 non-null  float64
 8   sleep_hours     88456 non-null  float64
 9   deep_sleep_pct  88456 non-null  float64
 10  resting_hr      88456 non-null  float64
 11  hrv_ms          80468 non-null  float64
 12  stress_score    88456 non-null  float64
 13  calories        88456 non-null  float64
 14  age_group       88456 non-null  str    
 15  lat             88456 non-null  float64
 16  lon             88456 non-null  float64
dtypes: float64(9), int64(2), str(6)
memory usage: 1

## CI of a mean: average resting heart rate

* Step 1: take a sample of the feature n=
* Step 2: ask scipy for 95 % CI t.intervalmethod
* Step 3: did we catch the truth?

In [8]:
## step 1
rng = np.random.default_rng(6)

sample = rng.choice(data["resting_hr"], size=50, replace=False)
print("sample mean:", sample.mean().round(2))

sample mean: 71.12


In [11]:
## step 2
ci = stats.t.interval(0.95, df =len(sample)-1, loc = sample.mean(), scale = stats.sem(sample))
print("95% CI", np.round(ci, 2))

95% CI [69.8  72.44]


In [13]:
len(data)

88456

In [14]:
## true mean of the population (assume the population is the full dataset)

data["resting_hr"].mean()

np.float64(71.73121099755811)

## confidence interval for a proportion

* let us think of the users that do more than 10_000 steps (desired goal)
* a success is when a user has a step count bigger than 10_0000


### CI approach

* Step 1: sample a certain number of days  n = len(data), k= sample
* Step 2: success from a cerzain population on a sample


df["success] = df["steps].apply(lambda x: 1 if x > 10000, else 0) Bernoulli(p) p prob success


#####. k samples, n size dataset

$P(X = k) = C^n_{k} p^k (1-p)^{n-k}$

Binomial (n,k)





In [40]:
## step 1: create samples
sample_size = 10000

days = rng.choice(data["steps"], size = sample_size, replace = False)
## successes or hits
hits = (days > 10000).sum()
print(f"{hits} of 400 days -> {hits/sample_size:.1%}")

### 26% of the 400 random sampled days we observe that a user walked more than 10000 steps

2549 of 400 days -> 25.5%


In [41]:
## step 2: CI

ci = stats.binomtest(hits, n=sample_size).proportion_ci(confidence_level=0.95)
print(f"the CI for the prob of someone walking more than 10000 steps with 95% confidence is {ci.low} and {ci.high}")

the CI for the prob of someone walking more than 10000 steps with 95% confidence is 0.2463807306162169 and 0.2635622264866761


## A/B test

ab

In [44]:
ab = pd.read_csv("../data/ab_test.csv")
ab

,a,b
0,0.27,13.61
1,6.08,21.53
2,13.74,9.23
3,9.70,5.36
4,7.00,12.90
5,4.77,10.83
6,2.56,10.70
7,7.24,15.57
8,7.30,10.42
9,0.86,18.90


## A/ B testing framework 

* Step 0: set the hypothesis before we look at any data
* Step 1: Run the experiment: sample randomly the data to create the two groups
* Step 2: we build the contingency table making the counts of conversions
* Step 3: calculate the p value 
* Step 4: decide a certain alpha: p_value < alpha -> reject H0


### null hypothesis

* H0: which group has higher conversion

In [45]:
ab.describe()

,a,b
count,50.00000,50.000000
mean,9.35540,12.466000
std,5.23527,6.491255
min,0.27000,0.590000
25%,5.42000,8.112500
50%,9.07000,12.635000
75%,13.46000,17.850000
max,20.90000,24.000000


#### comments on the statistics
* in average group b spends more time to convert than group a (around 3.1 min)
* SE = s /sqrt(50)
* t test (is for 2 different groups sampled from a certain population where we measure prob success)


In [47]:
result = stats.ttest_ind(ab["b"], ab["a"], equal_var = False)
result

TtestResult(statistic=np.float64(2.637533181209767), pvalue=np.float64(0.009776243024828829), df=np.float64(93.79317907867807))

In [49]:
print(f"t = {result.statistic:.2f}, p_value = {result.pvalue:.4f}")

t = 2.64, p_value = 0.0098


In [ ]:
### the degree significance of my test is 5%
p_value = result.pvalue
p_value < 0.05 ### reject the H0

np.True_